<a href="https://colab.research.google.com/github/IDishaSinghal/Judicial_Document_Intelligence_System/blob/main/OCRndLayoutLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pytesseract
!pip install opencv-python

In [2]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 30.3 MB/s eta 0:00:00


In [4]:
import cv2
import numpy as np
import easyocr
import os
import json

# ---------- PREPROCESSING ----------
def preprocess_for_easyocr(image_path):
    img = cv2.imread(image_path)

    # Step 1: Resize (important for OCR clarity)
    h, w = img.shape[:2]
    scale = 1200 / min(h, w)
    img = cv2.resize(img, None, fx=scale, fy=scale,
                     interpolation=cv2.INTER_CUBIC)

    # Step 2: Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Step 3: CLAHE (boost contrast safely)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Step 4: Mild denoising (DON’T overdo)
    denoised = cv2.fastNlMeansDenoising(enhanced, None, 10, 7, 21)

    # Step 5: Light sharpening (subtle)
    kernel = np.array([[0, -1, 0],
                       [-1, 4.5, -1],
                       [0, -1, 0]])
    sharpened = cv2.filter2D(denoised, -1, kernel)

    # ❌ NO thresholding (this was killing you)
    return sharpened


# ---------- OCR ----------
reader = easyocr.Reader(['en', 'hi'], gpu=True)

def run_ocr(image_path, conf_threshold=0.2):
    img = preprocess_for_easyocr(image_path)

    results = reader.readtext(img)

    extracted = []
    for (bbox, text, confidence) in results:
        # Filter garbage but not too aggressively
        if confidence >= conf_threshold and len(text.strip()) > 1:
            extracted.append({
                'text': text.strip(),
                'confidence': float(confidence),           # ✅ np.float64 → float
                'bbox': [[int(c) for c in pt] for pt in bbox]
            })

    return extracted

folder_path='/content/drive/MyDrive/Train'
output_path='/content/drive/MyDrive/Judiciary_train_results/train_results.json.txt'

# ---------- RUN ----------
image_files = [f for f in os.listdir(folder_path)]

all_results = {}

for i, fname in enumerate(image_files, 1):
    full_path = os.path.join(folder_path, fname)

    try:
        results = run_ocr(full_path)
        print(f"[{i}] ✅ {fname}: {len(results)} detections")
        all_results[f"file_{i}_{fname}"] = results

    except Exception as e:
        print(f"[{i}] ❌ {fname}: {e}")
        all_results[f"file_{i}_{fname}"] = {"error": str(e), "file": fname}

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("done")

[1] ✅ img_010.png: 2 detections
[2] ✅ img_31.png: 3 detections
[3] ✅ img_025.PNG: 7 detections
[4] ✅ img_016.jpg: 2 detections
[5] ❌ img_008.png: 'NoneType' object has no attribute 'shape'
[6] ✅ img_011.jpg: 5 detections
[7] ✅ img_004.jpg: 9 detections
[8] ✅ img_009.png: 3 detections
[9] ✅ img_007.png: 5 detections
[10] ✅ img_030.png: 3 detections
[11] ✅ img_019.jpg: 10 detections
[12] ✅ img_017.jpg: 2 detections
[13] ✅ img_012.jpg: 5 detections
[14] ❌ img_024.PNG: 'NoneType' object has no attribute 'shape'
[15] ✅ img_014.png: 2 detections
[16] ✅ img_002.jpg: 17 detections
[17] ✅ img_023.JPG: 18 detections
[18] ✅ img_021.PNG: 12 detections
[19] ✅ img_003.jpg: 15 detections
[20] ✅ img_018.jpg: 0 detections
[21] ✅ img_022.JPG: 0 detections
[22] ✅ img_026.JPG: 4 detections
[23] ✅ img_028.JPG: 7 detections
[24] ✅ img_015.png: 5 detections
[25] ✅ img_027.PNG: 6 detections
[26] ✅ img_029.JPG: 13 detections
[27] ✅ img_020.jpg: 21 detections
[28] ✅ img_013.jpg: 5 detections
[29] ✅ img_006.png: